# 02-chunk · 01 — character splitting (the baseline that gets beaten)

This notebook implements a fixed-size chunker (`chunks_fixed`,
`MEDICAL_SEPARATORS`, a references-stripping loop, and an `--adaptive`
sentence-similarity path) — the straightforward baseline chunking strategy
before anything structure-aware enters the picture.

**Scope.** The chunking functions below are self-contained: no external
ingest-pipeline coupling, no Pinecone/S3 upsert, no CLI. Store backends are
notebook `03`'s job; this notebook only produces chunk records.

**The point of this notebook.** `RecursiveCharacterTextSplitter` splits on a
word budget with a separator preference order — it has no idea a markdown
table exists. The last cell below builds a small synthetic outcomes table,
runs it through this chunker, and shows the table getting cut across a chunk
boundary, with the column header stranded in the earlier chunk. Notebook `02`
runs the *same* synthetic table through Docling's `HybridChunker` and the
table survives as one unit — that contrast is the entire argument for this
stage.

## What this notebook builds

| Name | What it does | Example |
| --- | --- | --- |
| `MEDICAL_SEPARATORS` | Ordered separator list (section headers, then paragraph/sentence/word) fed to `RecursiveCharacterTextSplitter` | splits on `"\nResults"` before falling back to `"\n\n"` |
| `chunks_fixed(text)` | Fixed-size chunker: strips a trailing References section, then splits on `MEDICAL_SEPARATORS` at a ~400-word budget | `chunks_fixed(synthetic_text)` -> a table cut across two chunks |
| `_get_sentences(text)` | Splits raw text into sentences on `.!?` boundaries | `_get_sentences("A. B.")` -> `["A.", "B."]` |
| `_cosine_similarity(a, b)` | Cosine similarity between two embedding vectors | `_cosine_similarity([1,0], [1,0])` -> `1.0` |
| `_adaptive_chunk_groups(sentences, embeddings)` | Groups adjacent sentence indices whose embeddings clear a similarity threshold | 5 sentences -> `[[0,1],[2],[3,4]]` |
| `chunks_adaptive(text, embed_fn)` | Sentence-split -> embed -> cosine-group -> merge; a chunk boundary falls where meaning shifts, not at a fixed word count | `chunks_adaptive(text, hash_embed_stub)` |
| `hash_embed_stub(texts)` | Deterministic, offline, non-semantic embedding stand-in so `chunks_adaptive` runs with no API key | `hash_embed_stub(["a", "b"])` -> two 16-dim vectors |


## Step 1 — set up the environment

Put the repo root on `sys.path` so `import nbio` works no matter where Jupyter started the kernel, then run `nbio.bootstrap()`.

In [ ]:
import sys
from pathlib import Path

# The kernel's cwd is this notebook's own directory (that's how Jupyter
# starts kernels), not the repo root -- so a bare `import nbio` fails two
# directories down unless the repo root goes on sys.path first. Same
# walk-up nbio.py's own bootstrap() uses internally.
_root = Path.cwd().resolve()
for _ in range(6):
    if (_root / "nbio.py").is_file():
        break
    _root = _root.parent
else:
    raise RuntimeError("could not locate nbio.py above the current directory")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import nbio
nbio.bootstrap()


## Step 2 — define the medical-aware separator list and splitter config

`CHUNK_SIZE`/`CHUNK_OVERLAP` are word counts (the splitter's `length_function` counts words, not characters). `MEDICAL_SEPARATORS` is tried in order: section headers first, then inline section headers that show up mid-sentence in poorly-segmented text, then the ordinary paragraph/sentence/word fallbacks. Nothing in this list knows what a table row looks like — that gap is the point of this notebook.

In [ ]:
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter

CHUNK_SIZE = 400
CHUNK_OVERLAP = 50
MIN_CHUNK_WORDS = 50

# Medical paper section headers as separators (split on these in order before
# falling back to \n\n, \n, " ").
MEDICAL_SEPARATORS = [
    "\nAbstract", "\nABSTRACT",
    "\nBackground", "\nBACKGROUND",
    "\nIntroduction", "\nINTRODUCTION",
    "\nMethods", "\nMETHODS",
    "\nMaterials and Methods",
    "\nMaterials and methods",
    "\nResults", "\nRESULTS",
    "\nResults and Discussion",
    "\nDiscussion", "\nDISCUSSION",
    "\nConclusion", "\nCONCLUSION",
    "\nConclusions", "\nCONCLUSIONS",
    "\nReferences", "\nREFERENCES",
    "\nAcknowledg",
    # Inline section headers inside sentences (e.g. "...fish skin. Introduction
    # Fish skin has been used...") — real OCR/extraction text often loses the
    # paragraph break in front of a heading.
    ". Introduction ",
    ". Methods ",
    ". Results ",
    ". Discussion ",
    ". Conclusion ",
    "\n\n",
    "\n",
    ". ",
    " ",
]

_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=MEDICAL_SEPARATORS,
    length_function=lambda t: len(t.split()),
)


## Step 3 — define `chunks_fixed`, the fixed-size chunker

Strips a trailing References section first (references are not evidence and only dilute retrieval), then hands the rest to the separator-ordered splitter above.

In [ ]:
def chunks_fixed(text: str) -> list[str]:
    """Chunk using RecursiveCharacterTextSplitter with medical section
    separators (~400 words per chunk). Strips a trailing References section
    first — references are not evidence and only dilute retrieval.
    """
    if not text or len(text.split()) < MIN_CHUNK_WORDS:
        return [text] if text and text.strip() else []

    for marker in ["\nReferences\n", "\nREFERENCES\n", "\nReferences ", "\nREFERENCES "]:
        idx = text.find(marker)
        if idx > 0:
            text = text[:idx]
            break

    chunks = _splitter.split_text(text)
    return [c for c in chunks if len(c.split()) >= MIN_CHUNK_WORDS]


## Step 4 — split text into sentences

Adaptive chunking needs sentence boundaries before anything else: `_get_sentences` splits raw text on `.!?` boundaries, capped at 150 sentences so a runaway document can't blow up the embedding step below.

In [ ]:
import hashlib
import math


def _get_sentences(text: str, max_sentences: int = 150) -> list[str]:
    """Split text into sentences. Raw text, no cleaning."""
    if not text or not text.strip():
        return []
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in parts if s.strip()][:max_sentences]




## Step 5 — cosine similarity between two embedding vectors

The primitive `_adaptive_chunk_groups` needs next: how similar are two adjacent sentences once embedded.

In [ ]:
def _cosine_similarity(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(x * x for x in b))
    if na == 0 or nb == 0:
        return 0.0
    return dot / (na * nb)




## Step 6 — group adjacent sentences by similarity

Walk the sentence list once; keep extending the current group while adjacent cosine similarity clears `min_similarity`, otherwise start a new group. This is the actual chunk-boundary decision — it falls wherever two consecutive sentences stop being "about the same thing."

In [ ]:
def _adaptive_chunk_groups(sentences: list[str], embeddings: list[list[float]],
                            min_similarity: float = 0.6) -> list[list[int]]:
    """Group adjacent sentence indices whose cosine similarity clears the
    threshold."""
    if not sentences or not embeddings:
        return []
    if len(sentences) != len(embeddings):
        return [[i] for i in range(len(sentences))]

    groups, current = [], [0]
    for i in range(1, len(sentences)):
        sim = _cosine_similarity(embeddings[i - 1], embeddings[i])
        if sim >= min_similarity:
            current.append(i)
        else:
            groups.append(current)
            current = [i]
    groups.append(current)
    return groups




## Step 7 — `chunks_adaptive`, the similarity-based chunker

Ties the previous three steps together: sentence-split, embed (via a pluggable `embed_fn` — picking a real embedding model is stage 03's job, not this stage's), group by similarity, merge each group back into a chunk.

In [ ]:
def chunks_adaptive(text: str, embed_fn, min_chunk_words: int = MIN_CHUNK_WORDS) -> list[str]:
    """Adaptive chunking: sentence split -> embed -> cosine group -> merge."""
    sentences = _get_sentences(text)
    if not sentences:
        return [text] if text.strip() else []
    if len(sentences) == 1:
        return [text] if len(text.split()) >= min_chunk_words else []

    embeddings = embed_fn(sentences)
    groups = _adaptive_chunk_groups(sentences, embeddings)
    chunks = []
    for g in groups:
        chunk = " ".join(sentences[i] for i in g)
        if len(chunk.split()) >= min_chunk_words:
            chunks.append(chunk)
    return chunks if chunks else [text]




## Step 8 — an offline stand-in for `embed_fn`

To keep this notebook runnable with no API key, `hash_embed_stub` plugs in a deterministic hash-based embedding — the same offline path used elsewhere in this stage. **The resulting "embeddings" are not semantically meaningful**; they exist only to prove the grouping mechanics run end to end.

In [ ]:
def hash_embed_stub(texts: list[str], dim: int = 16) -> list[list[float]]:
    """Deterministic, offline, NOT semantically meaningful. Exists only so
    chunks_adaptive() has something to call without a real API key — swap for
    a real embedding call (stage 03) to get real similarity grouping."""
    out = []
    for t in texts:
        h = hashlib.sha256(t.encode("utf-8")).digest()
        vec = [b / 255.0 for b in h[:dim]]
        out.append(vec)
    return out


## Step 9 — build the synthetic outcomes table

A small "Results" section: six repeats of one sentence (tuned so the section crosses the 400-word budget partway through the table), then a five-row markdown outcomes table, then one closing sentence. Nothing here is real patient data — every value is invented for this demonstration.

In [ ]:
prose_sentence = (
    "Patients were followed for twelve months after the index procedure to "
    "record wound healing time and complication rates in each treatment arm. "
)
prose = (prose_sentence * 14).strip()

table_rows = [
    "| Patient ID | Treatment Arm | Wound Healing (days) | Complication |",
    "| --- | --- | --- | --- |",
    "| P001 | Early excision | 14 | None |",
    "| P002 | Delayed excision | 21 | Infection |",
    "| P003 | Early excision | 12 | None |",
    "| P004 | Delayed excision | 25 | Graft failure |",
    "| P005 | Early excision | 15 | None |",
    "| P006 | Delayed excision | 19 | Infection |",
    "| P007 | Early excision | 13 | None |",
    "| P008 | Delayed excision | 23 | None |",
]
table = "Outcomes Table\n" + "\n".join(table_rows)
trailing = "These results are discussed further in the next section."

synthetic_text = "\nResults\n" + prose + "\n" + table + "\n" + trailing
print(f"{len(synthetic_text.split())} words total")


## Step 10 — run `chunks_fixed` on the table and look at the cut

This is the actual point of the notebook: run the fixed-size chunker on the synthetic table and check which chunk (if any) keeps the column header.

In [ ]:
result_chunks = chunks_fixed(synthetic_text)
print(f"{len(result_chunks)} chunks\n")
for i, c in enumerate(result_chunks):
    has_header = "Patient ID" in c
    print(f"--- chunk {i}  ({len(c.split())} words)  header row present: {has_header} ---")
    print(c)
    print()


## What happened

Chunk 0 carries the table's column header (`Patient ID | Treatment Arm | ...`)
and the first six data rows, then the 400-word budget is hit and the splitter
cuts. Chunk 1 re-opens (with the configured 50-word overlap) partway through
the row list and runs to the end of the table plus the trailing sentence —
**with no column header anywhere in it**. Read chunk 1 alone and the numbers
in it are uninterpretable: which column is which row's `14`? A retrieval
system that returns chunk 1 for "what was the wound healing time in the
early-excision arm" has already lost the answer.

This is not a bug in the splitter — `RecursiveCharacterTextSplitter` is doing
exactly what it is configured to do: pack text into `chunk_size`-sized pieces
using the best available separator. It has no representation of "table," "row"
or "header," so it cannot protect a boundary it cannot see. Notebook `02` runs
the identical synthetic table through a chunker that *does* have that
representation.
